# 04 · BPE a mano · el cuchillo del modelo

**Parte II · El chef ya existe.** En la Parte I un token era **una letra**. Qwen no ve letras: ve pedazos como `▁la`, `ción` o `▁Mancha`. Aquí programas el algoritmo que fabrica esos pedazos (**BPE**, *Byte Pair Encoding*) en Python puro, sobre el mismo Quijote. Sin torch, sin GPU.

Al final vas a poder contestar tres preguntas de examen:

1. ¿De dónde salen los tokens? (Nadie los escribió: se **entrenan**.)
2. ¿Por qué un modelo con ventana de 4 096 tokens "lee" más que uno de 4 096 letras?
3. ¿Por qué no puedes cambiarle el tokenizer a Qwen sin romperlo?

## Dummy (antes de la fórmula)

Imagina que escribes el Quijote a mano y te cansas. Notas que `q` casi siempre va con `ue`. Inventas un sello `que` y lo usas cada vez. Luego notas que `▁d` + `e` aparece muchísimo (`de`, `del`, `dejar`…) y haces otro sello. Repites **mil veces**: siempre sellas la pareja **más frecuente** de lo que ya tienes.

Eso es BPE. Al final tienes:

- una **lista de sellos en orden** (los *merges*), y
- un **vocabulario**: las letras de base + un sello nuevo por merge.

Para cortar un texto nuevo, aplicas los sellos **en el mismo orden** en que los aprendiste. Palabras comunes quedan en un solo pedazo; palabras raras se parten en pedazos más chicos, pero **nunca** se vuelven imposibles de escribir.

## Matemática (la misma idea, en serio)

Sea $w = (s_1,\dots,s_n)$ una palabra como secuencia de símbolos y $c(w)$ cuántas veces aparece en el corpus. La cuenta de un par es

$$
\mathrm{cuenta}(a,b) \;=\; \sum_{w} c(w)\,\bigl|\{\,i : s_i=a,\ s_{i+1}=b\,\}\bigr|.
$$

En el paso $k$ se elige $(a,b)^\star_k = \arg\max_{(a,b)} \mathrm{cuenta}(a,b)$ y en **todas** las palabras se reemplaza la pareja contigua $a\,b$ por un símbolo nuevo $ab$. Es un algoritmo **codicioso**: no busca el vocabulario óptimo, solo el mejor paso siguiente.

Con $K$ merges, $|V| = |V_0| + K$. Si $\bar\ell$ = caracteres por token, una ventana de $T$ tokens cubre $\approx T\,\bar\ell$ caracteres. Subir $K$ sube $\bar\ell$ (ventana más "larga") pero también sube $|V|$: la capa de salida es una matriz $d\times|V|$ y el softmax reparte masa entre más opciones.

**El tokenizer es parte de $\theta$.** El embedding $E\in\mathbb{R}^{|V|\times d}$ tiene un renglón por token. Si cambias los merges, el renglón 1 734 ya no significa lo mismo. Por eso el tokenizer viaja **con** el modelo.

## 0 · El corpus

In [1]:
# Colab: descomenta la línea de abajo (una vez). Este notebook no necesita torch.
# !wget -q https://raw.githubusercontent.com/NoSoyRo/diplomado-ia-llms/main/datos/quijote.txt

import collections
import math
import re
import time
from pathlib import Path

TEXT_PATH = next(
    (p for p in (Path("quijote.txt"), Path("../datos/quijote.txt"), Path("datos/quijote.txt")) if p.exists()),
    Path("quijote.txt"),
)
texto = TEXT_PATH.read_text(encoding="utf-8").replace("\r\n", "\n")

ENTRENO = texto[:400_000]          # aquí aprende los sellos
PRUEBA = texto[1_200_000:1_260_000]  # aquí los medimos (texto que no vio)
print(f"corpus={len(texto):,} chars · entreno={len(ENTRENO):,} · prueba={len(PRUEBA):,}")

corpus=2,097,953 chars · entreno=400,000 · prueba=60,000


## 1 · Pre-tokenizar: no se fusiona a través de palabras

Antes de BPE se corta el texto en "palabras" con su espacio **pegado al inicio** (`" lugar"`, no `"lugar"`). Así `▁de` (inicio de palabra) y `de` (dentro de `puede`) son tokens distintos, y ningún merge cruza dos palabras. GPT-2, Qwen y Llama hacen algo parecido con una regex más larga.

Para verlo usamos `▁` en lugar del espacio y `↵` en lugar del salto de línea.

In [2]:
PRE = re.compile(r" ?[^\W\d_]+| ?\d+| ?[^\s\w]+|\s+")


def pretokenizar(s: str) -> list[str]:
    return PRE.findall(s)


def ver(tokens) -> str:
    return " | ".join(t.replace(" ", "▁").replace("\n", "↵") for t in tokens)


print(ver(pretokenizar("En un lugar de la Mancha, de cuyo nombre no quiero acordarme")))

En | ▁un | ▁lugar | ▁de | ▁la | ▁Mancha | , | ▁de | ▁cuyo | ▁nombre | ▁no | ▁quiero | ▁acordarme


## 2 · Entrenar: contar parejas, sellar la ganadora, repetir

Truco de eficiencia (el mismo que usan las librerías): no recorres el texto 1 000 veces. Cuentas cada palabra **una vez** (`Counter`) y trabajas sobre ~11 mil palabras únicas con su frecuencia $c(w)$.

In [3]:
Palabra = tuple[str, ...]


def fusionar(w: Palabra, a: str, b: str) -> Palabra:
    """Reemplaza cada pareja contigua (a, b) por el símbolo a+b."""
    out, i = [], 0
    while i < len(w):
        if i < len(w) - 1 and w[i] == a and w[i + 1] == b:
            out.append(a + b)
            i += 2
        else:
            out.append(w[i])
            i += 1
    return tuple(out)


def entrenar_bpe(s: str, n_merges: int) -> list[tuple[str, str]]:
    palabras: dict[Palabra, int] = {tuple(w): c for w, c in collections.Counter(pretokenizar(s)).items()}
    merges: list[tuple[str, str]] = []
    for _ in range(n_merges):
        pares: collections.Counter = collections.Counter()
        for w, c in palabras.items():
            for par in zip(w, w[1:]):
                pares[par] += c            # cuenta(a, b) = Σ c(w) · apariciones
        if not pares:
            break
        (a, b), _ = pares.most_common(1)[0]   # argmax: el paso codicioso
        merges.append((a, b))
        palabras = {(fusionar(w, a, b) if a in w else w): c for w, c in palabras.items()}
    return merges


t0 = time.time()
MERGES = entrenar_bpe(ENTRENO, 1000)
print(f"{len(MERGES)} merges en {time.time() - t0:.1f}s")
print("primeros 12:", ver(a + b for a, b in MERGES[:12]))
print("últimos 12: ", ver(a + b for a, b in MERGES[-12:]))

1000 merges en 6.9s
primeros 12: ▁d | ue | en | ▁l | ▁a | es | que | os | ▁s | er | ▁c | ▁de
últimos 12:  ▁len | ▁acer | tir | ▁estu | ▁Dulcinea | ▁pies | boso | ▁tomar | ele | ▁aventuras | ▁quiso | ▁debía


**Lee los primeros merges.** Son `▁d`, `ue`, `en`, `▁l`… las parejas del español más frecuentes. Nadie le dijo al algoritmo qué es una sílaba ni qué es `que`: salió de contar. Los últimos ya son pedazos de palabra enteros (`miento`, `▁dicho`).

## 3 · Codificar y decodificar

Codificar un texto nuevo = pre-tokenizar y, dentro de cada palabra, aplicar el merge con **menor rango** (el que se aprendió primero) mientras haya alguno. Decodificar es pegar los pedazos: no se pierde nada.

In [4]:
def codificar(s: str, merges: list[tuple[str, str]]) -> list[str]:
    rango = {par: i for i, par in enumerate(merges)}
    tokens: list[str] = []
    for pal in pretokenizar(s):
        w = list(pal)
        while len(w) > 1:
            r, i = min((rango.get((w[j], w[j + 1]), math.inf), j) for j in range(len(w) - 1))
            if r == math.inf:
                break
            w[i : i + 2] = [w[i] + w[i + 1]]
        tokens.extend(w)
    return tokens


def decodificar(tokens: list[str]) -> str:
    return "".join(tokens)


frase = "En un lugar de la Mancha, de cuyo nombre no quiero acordarme"
for k in (0, 50, 200, 1000):
    toks = codificar(frase, MERGES[:k])
    print(f"merges={k:>4} · {len(toks):>2} tokens · {ver(toks)}")

assert decodificar(codificar(PRUEBA, MERGES)) == PRUEBA
print("\ndecodificar(codificar(x)) == x  ✔  (sin pérdida)")

merges=   0 · 60 tokens · E | n | ▁ | u | n | ▁ | l | u | g | a | r | ▁ | d | e | ▁ | l | a | ▁ | M | a | n | c | h | a | , | ▁ | d | e | ▁ | c | u | y | o | ▁ | n | o | m | b | r | e | ▁ | n | o | ▁ | q | u | i | e | r | o | ▁ | a | c | o | r | d | a | r | m | e
merges=  50 · 38 tokens · E | n | ▁ | un | ▁l | u | g | ar | ▁de | ▁la | ▁ | M | an | ch | a | , | ▁de | ▁c | u | y | o | ▁no | m | b | re | ▁no | ▁ | q | ui | er | o | ▁a | c | or | d | ar | m | e
merges= 200 · 30 tokens · E | n | ▁un | ▁l | u | g | ar | ▁de | ▁la | ▁ | M | an | ch | a | , | ▁de | ▁cu | y | o | ▁no | m | bre | ▁no | ▁qui | ero | ▁ac | or | d | ar | me
merges=1000 · 17 tokens · En | ▁un | ▁lugar | ▁de | ▁la | ▁Mancha | , | ▁de | ▁cu | yo | ▁nombre | ▁no | ▁quiero | ▁ac | or | dar | me

decodificar(codificar(x)) == x  ✔  (sin pérdida)


Fíjate que `MERGES[:k]` también es un tokenizer válido: BPE aprende en orden, así que cortar la lista es "entrenar menos". Lo usamos para medir.

## 4 · ¿Cuánto texto cabe en la ventana?

Medimos $\bar\ell$ = caracteres por token **en texto que no vio**, para distintos $K$. Con la ventana `seq_len = 64` de la Parte I, ¿cuántas letras de Quijote "ve" el modelo?

In [5]:
SEQ_LEN = 64
print(f"{'merges':>7} {'|V|':>6} {'tokens':>8} {'chars/token':>12} {'chars en ventana de 64':>24}")
base = sorted(set(ENTRENO))
for k in (0, 50, 100, 250, 500, 1000):
    n_tok = len(codificar(PRUEBA, MERGES[:k]))
    ell = len(PRUEBA) / n_tok
    print(f"{k:>7} {len(base) + k:>6} {n_tok:>8,} {ell:>12.2f} {SEQ_LEN * ell:>24.0f}")

 merges    |V|   tokens  chars/token   chars en ventana de 64
      0     83   60,000         1.00                       64
     50    133   38,474         1.56                      100
    100    183   33,609         1.79                      114
    250    333   27,793         2.16                      138
    500    583   24,137         2.49                      159
   1000   1083   21,221         2.83                      181


**Cómo leer la tabla.** Con 0 merges es el modelo de la Parte I: 1 char/token, la ventana de 64 ve 64 letras. Con 1 000 merges cada token carga casi 3 letras: la **misma** ventana ve casi 3× más Quijote. Un tokenizer real (Qwen: ~151 mil tokens) saca más; la sección 6 lo mide en este mismo texto.

La ganancia se va aplanando: los primeros merges sellan parejas que salen en todas partes; los últimos sellan palabras que salen poco. Ese rendimiento decreciente es la razón de que nadie use un vocabulario de 10 millones.

## 5 · El cuchillo del Quijote corta mal el 2026

El tokenizer refleja **su** corpus. Mira cómo corta palabras que Cervantes nunca escribió.

In [6]:
for s in (" la Mancha", " caballero andante", " Banxico mantiene la tasa", " inteligencia artificial", " streaming"):
    toks = codificar(s, MERGES)
    print(f"{len(toks):>2} tokens · {ver(toks)}")

 2 tokens · ▁la | ▁Mancha
 2 tokens · ▁caballero | ▁andante
12 tokens · ▁B | an | x | i | co | ▁man | t | iene | ▁la | ▁t | as | a
10 tokens · ▁in | t | el | ig | encia | ▁ar | ti | fi | ci | al
 5 tokens · ▁s | tre | am | in | g


`▁Mancha` sale en pocos pedazos; `Banxico` o `streaming` se hacen trizas. Más pedazos = más pasos de next-token para la misma palabra, y cada pedazo tiene un embedding con poca práctica. Por eso los tokenizers serios se entrenan con texto de **todo**: web, código, varios idiomas.

### ¿Y lo que nunca vio?

Nuestro vocabulario base son las letras del Quijote. Un emoji no está ahí: no tiene id. Los tokenizers modernos son **byte-level**: la base son los 256 bytes posibles, así que cualquier texto (UTF-8) se puede escribir, aunque sea byte por byte.

In [7]:
VOCAB = {t: i for i, t in enumerate(sorted(set(ENTRENO)) + [a + b for a, b in MERGES])}
print(f"|V| = {len(VOCAB)}")

for s in (" la Mancha", " ¡ya llegó el 🙂!"):
    toks = codificar(s, MERGES)
    faltan = [t for t in toks if t not in VOCAB]
    print(f"{s!r:24} → ids {[VOCAB.get(t, '??') for t in toks]}   sin id: {faltan}")

print("\nbyte-level: '🙂' =", list("🙂".encode("utf-8")), "→ 4 bytes, todos en 0..255: siempre hay id")

|V| = 1083
' la Mancha'             → ids [114, 955]   sin id: []
' ¡ya llegó el 🙂!'       → ids [812, 745, 930, 116, 1, '??', 2]   sin id: ['🙂']

byte-level: '🙂' = [240, 159, 153, 130] → 4 bytes, todos en 0..255: siempre hay id


## 6 · (Opcional) El cuchillo de verdad: Qwen

Si tienes `transformers` instalado, compara con el tokenizer de `Qwen/Qwen2.5-0.5B-Instruct` (baja ~7 MB, no el modelo). Si no, la celda solo avisa.

In [8]:
try:
    from transformers import AutoTokenizer

    qwen = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
    for s in (frase, " Banxico mantiene la tasa", " ¡ya llegó el 🙂!"):
        ids = qwen.encode(s)
        print(f"{len(ids):>2} tokens · {ver(qwen.convert_ids_to_tokens(ids))}")
    n_q = len(qwen.encode(PRUEBA))
    print(f"\nQwen: |V|={len(qwen):,} · {len(PRUEBA) / n_q:.2f} chars/token en el mismo texto de prueba")
except Exception as exc:  # sin transformers o sin red
    print(f"(se salta: {type(exc).__name__}) — pip install transformers para comparar con Qwen")

/Users/rod/clases/diplomado-ia-llms/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


17 tokens · En | Ġun | Ġlugar | Ġde | Ġla | ĠMan | cha | , | Ġde | Ġc | uyo | Ġnombre | Ġno | Ġquiero | Ġac | ord | arme
 8 tokens · ĠBan | xico | Ġm | anti | ene | Ġla | Ġt | asa
 6 tokens · ĠÂ¡ | ya | ĠllegÃ³ | Ġel | ĠðŁĻĤ | !

Qwen: |V|=151,665 · 3.20 chars/token en el mismo texto de prueba


Tres cosas que ver:

- `Ġ` es como Qwen pinta el byte del espacio (el mismo papel que nuestro `▁`).
- Qwen corta `Banxico mantiene la tasa` en menos pedazos que el nuestro: vio mucho español moderno. Pero parte `▁Mancha` en dos, y el nuestro la deja entera. El nuestro es **especialista** en Quijote; Qwen es **generalista**.
- `Â¡`, `Ã³`, `ðŁĻĤ` no son basura: son los bytes UTF-8 pintados con el mapeo byte→carácter de GPT-2. `ó` son 2 bytes; el emoji son 4 bytes y aun así cabe en **un** token. Eso es byte-level: nada se queda sin id.

---

## Entregable

1. La tabla de la sección 4, y una frase: ¿por qué la ganancia de chars/token se aplana?
2. Una palabra moderna que el BPE del Quijote corte en ≥ 4 pedazos, y cuántos le da Qwen.
3. Contesta en tus palabras: *si entreno un LoRA sobre Qwen, ¿puedo usar mi BPE del Quijote como tokenizer?* (Pista: renglones de $E$.)

## Para romperlo

- Entrena con `texto[:100_000]` vs `texto[:1_600_000]`. ¿Cambian los primeros merges? ¿Y los últimos?
- Quita la pre-tokenización (`pretokenizar = lambda s: [s]`, con menos texto). ¿Qué merges aparecen que cruzan palabras? ¿Por qué eso es mala idea?
- Haz la versión byte-level: `w = tuple(pal.encode("utf-8"))` y merges sobre bytes. Ahora el emoji sí tiene id.